# Nifty-50 MTL stock recommendation — key experiments

This notebook reproduces the tables and figures of the paper from the artefacts written by `scripts/`. Run the pipeline first (see README).

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
from nifty_mtl.config import RESULTS, TABLES, FIGURES, Config, load_universe
from nifty_mtl.features.build import FeatureSet
from nifty_mtl.data.preprocess import WeeklyPanel
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)
fs = FeatureSet.load(); panel = WeeklyPanel.load(); u = load_universe()
print(f'{len(fs.weeks)} weeks x {len(fs.symbols)} symbols, seq features {fs.seq.shape[-1]}, static {fs.static.shape[-1]}, usable samples {int(fs.sample_ok.sum())}')

## 1. Data quality and universe

In [ ]:
q = pd.read_csv(TABLES / 'data_quality.csv', index_col=0) if (TABLES/'data_quality.csv').exists() else None
display(q.sort_values('valid_frac').head(8) if q is not None else 'run `nifty-mtl data`')
yr, yv = panel.targets(); print('vol target quantiles:', yv[panel.valid].stack().quantile([.05,.25,.5,.75,.95]).round(4).to_dict())

## 2. Hyper-parameter search

In [ ]:
import optuna, warnings; warnings.filterwarnings('ignore')
st = optuna.load_study(study_name='mtl_v2', storage=f'sqlite:///{RESULTS}/optuna.db')
df = st.trials_dataframe().sort_values('value', ascending=False)
df[['number','value','user_attrs_val_sharpe','params_lr','params_dropout','params_w_return','params_lambda_turnover','params_target_mode']].head(10)

## 3. Static model: training curves and IC by split

In [ ]:
display(pd.read_csv(TABLES / 'static_model_ic.csv', index_col=0).round(4))
display(Image(str(FIGURES / 'training_curves.png')))

## 4. Walk-forward results vs baselines

In [ ]:
main = pd.read_csv(TABLES / 'main_results.csv', index_col=[0,1])
display(main.loc['all_oos'].round(3)); display(main.loc['test'].round(3)); display(main.loc['holdout'].round(3))
for f in ['equity_curves','sharpe_ci','decile_returns','rolling_ic','drawdowns']: display(Image(str(FIGURES / f'{f}.png')))

## 5. Sensitivity and yearly breakdown

In [ ]:
display(pd.read_csv(TABLES / 'sensitivity.csv', index_col=0).round(3)); display(pd.read_csv(TABLES / 'yearly.csv', index_col=0).round(3))
display(Image(str(FIGURES / 'cost_sensitivity.png')))

## 6. Interpretability

In [ ]:
print(json.load(open(TABLES / 'shap_kpis.json'))); print(json.load(open(TABLES / 'attention_concentration.json')))
display(pd.read_csv(TABLES / 'shap_feature_importance.csv', index_col=0).head(15).round(4))
for f in ['attention_by_lag','shap_importance','shap_by_lag']: display(Image(str(FIGURES / f'{f}.png')))
display(pd.read_csv(TABLES / 'example_explanations.csv').head(10))

## 7. Failure analysis

In [ ]:
display(pd.read_csv(TABLES / 'failure_regimes.csv', index_col=[0,1]).round(3))
for by in ['sector','beta_bucket','size_bucket']: display(pd.read_csv(TABLES / f'failure_by_{by}.csv', index_col=0).round(3))
display(Image(str(FIGURES / 'failure_regimes.png')))